# Silver: atendentes

## Padronização
- `nivel_atendimento`: "1", "Nível 1", "N2", ... viram o número 1 ou 2.
- `status_atendente`: minúsculo e sem acento (ativo, inativo).
- `data_admissao` e `data_desligamento`: convertidas de DD/MM/YYYY para data.
- `nome_atendente`: tirar espaços sobrando.

## Regras de limpeza
1. Campo essencial ausente ou inválido (id, nome, nível, admissão, status): remove a linha.
2. `data_desligamento` preenchida, mas inválida: remove.
3. Admissão depois do desligamento: remove.
4. Tem data de desligamento, mas status ativo: remove.
5. Status inativo sem data de desligamento: remove.
6. ID repetido: cópias idênticas mantêm uma; dados diferentes anulam todas.

In [0]:
from pyspark.sql import functions as F, Window

catalog = "rocket_lab"
bronze_schema = f"{catalog}.vcredit_bronze"
silver_schema = f"{catalog}.vcredit_silver"
tabela_destino = f"{silver_schema}.atendentes"

# Padronização
COM = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM = "aaaaaeeeeiiiiooooouuuuc"

def normaliza(c):
    x = F.lower(F.trim(F.col(c)))
    x = F.translate(x, COM, SEM)
    x = F.regexp_replace(x, r"[\s\u00A0]+", " ")
    return F.upper(F.trim(x))

def para_data(c):
    # aceita dd/MM/yyyy e yyyy-MM-dd; texto inválido vira nulo
    return F.coalesce(
        F.expr(f"try_to_timestamp(trim({c}), 'dd/MM/yyyy')"),
        F.expr(f"try_to_timestamp(trim({c}), 'yyyy-MM-dd')"),
    ).cast("date")

In [0]:
df = spark.table(f"{bronze_schema}.atendentes")
n_bronze = df.count()

# Padronização das colunas
nivel_extraido = F.regexp_extract(normaliza("nivel_atendimento"), r"^(?:NIVEL ?|N ?)?([12])$", 1)
desl_texto = F.trim("data_desligamento")

df = (df
    .withColumn("id_atendente", F.trim("id_atendente"))
    .withColumn("nome_atendente", F.trim(F.regexp_replace("nome_atendente", r"[\s\u00A0]+", " ")))
    .withColumn("nivel_atendimento", F.when(nivel_extraido != "", nivel_extraido).cast("int"))
    .withColumn("status_atendente", F.lower(normaliza("status_atendente")))
    .withColumn("desl_preenchido",
                desl_texto.isNotNull() & ~F.upper(desl_texto).isin("", "NULL", "NAN", "NONE"))
    .withColumn("data_admissao", para_data("data_admissao"))
    .withColumn("data_desligamento", para_data("data_desligamento"))
)

# Função que remove as linhas de uma regra e conta quantas foram
contagem = {}
def remove_se(df, nome, cond):
    cond = F.coalesce(cond, F.lit(False))
    contagem[nome] = df.filter(cond).count()
    return df.filter(~cond)

# Regra 1: dados essenciais ausentes ou inválidos -> remove a linha
essencial_ok = F.coalesce(
    F.col("id_atendente").rlike(r"^\d+$")
    & (F.col("nome_atendente") != "")
    & F.col("nivel_atendimento").isNotNull()
    & F.col("data_admissao").isNotNull()
    & F.col("status_atendente").isin("ativo", "inativo"),
    F.lit(False))
df = remove_se(df, "1. campo essencial ausente ou inválido", ~essencial_ok)

# Regra 2: data_desligamento preenchida, mas não é uma data válida -> remove
df = remove_se(df, "2. data_desligamento preenchida e inválida",
               F.col("desl_preenchido") & F.col("data_desligamento").isNull())

# Regra 3: admissão depois do desligamento ("saiu antes de entrar") -> remove
df = remove_se(df, "3. admissão depois do desligamento",
               F.col("data_admissao") > F.col("data_desligamento"))

# Regra 4: tem data de desligamento mas status ativo -> remove
df = remove_se(df, "4. tem desligamento mas está ativo",
               F.col("data_desligamento").isNotNull() & (F.col("status_atendente") == "ativo"))

# Regra 5: status inativo mas sem data de desligamento -> remove
df = remove_se(df, "5. inativo sem data de desligamento",
               F.col("data_desligamento").isNull() & (F.col("status_atendente") == "inativo"))

# Regra 6a: cópias idênticas -> mantém uma só
cols_negocio = ["id_atendente", "nome_atendente", "nivel_atendimento",
                "data_admissao", "data_desligamento", "status_atendente"]
n_antes = df.count()
df = df.dropDuplicates(cols_negocio)
contagem["6a. cópias idênticas removidas"] = n_antes - df.count()

# Regra 6b: mesmo id_atendente com dados diferentes -> anula todas
w = Window.partitionBy("id_atendente")
df = df.withColumn("qtd", F.count("*").over(w))
contagem["6b. linhas anuladas por id conflitante"] = df.filter("qtd > 1").count()
df = df.filter("qtd = 1").drop("qtd")

# Transformar o id_atendente em inteiro e ordenar colunas
df = (df.withColumn("id_atendente", F.col("id_atendente").cast("int"))
        .select("id_atendente", "nome_atendente", "nivel_atendimento",
                "data_admissao", "data_desligamento", "status_atendente",
                "ingest_timestamp"))

# Gravação (lê da bronze e sobrescreve)
(df.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(tabela_destino))

# Resumo
n_silver = spark.table(tabela_destino).count()
print(f"bronze: {n_bronze} linhas")
for regra, n in contagem.items():
    print(f"  {regra}: {n}")
print(f"silver: {n_silver} linhas")

display(spark.table(tabela_destino).orderBy("id_atendente"))
display(spark.table(tabela_destino).groupBy("nivel_atendimento", "status_atendente").count())